# CS610 Integrated Risk Scoring Pipeline
**Pillar 1** (40%) — TF-IDF Identity Resolution  
**Pillar 2** (30%) — Crime Severity (K-Means cluster)  
**Pillar 4** (20%) — Hidden Linkage (GCN/GraphSAGE)  
**Pillar 5** (10%) — Visual Similarity (CLIP)  

---

### Workflow
1. Client submits a query (name + photo)
2. **P1**: TF-IDF cosine similarity finds closest fugitive match → `identity_score`
3. **P2**: Look up matched fugitive's crime type → `crime_severity`
4. **P4**: Look up matched fugitive's top-3 linked fugitives → weighted linkage score → `linkage_score`
5. **P5**: Look up visual similarity between client photo and matched fugitive → `visual_score`
6. **Final Risk** = P1×0.40 + P2×0.30 + P4×0.20 + P5×0.10

### P4 Formula
`P4 = (link1_score × severity1 × 0.50) + (link2_score × severity2 × 0.30) + (link3_score × severity3 × 0.20)`

## Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Load Data & Pre-Computed Assets

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
P4_DIR     = os.path.join(BASE_DIR, 'Pillar 4 (GCN and Graphsage)')


# ── Pillar 2: Crime severity from main dataset ─────────────────────────────────
CSV_PATH = os.path.join(P4_DIR, 'crime_analysis_results_aft_transformer_ner.csv')
df_fugitives = pd.read_csv(CSV_PATH)

# Deduplicate to one row per fugitive (keep first occurrence)
df_unique = df_fugitives.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
print(f'✅ Fugitive database loaded — {len(df_unique):,} unique fugitives')

# ── Pillar 1: Build TF-IDF char n-gram embeddings from df_unique ──────────────
# Rebuilt here to ensure index alignment with df_unique (1:1 mapping)
name_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 4),
    max_features=5000, sublinear_tf=True
)
name_embeddings = normalize(
    name_vectorizer.fit_transform(df_unique['name'].str.upper().astype(str))
)
print(f'✅ Pillar 1: TF-IDF embeddings built — shape: {name_embeddings.shape}')

# ── Pillar 4: Pre-computed top-3 links from GraphSAGE ─────────────────────────
P4_LINKS_PATH = os.path.join(P4_DIR, 'outputs', 'top_new_links_all.csv')
df_links = pd.read_csv(P4_LINKS_PATH)
print(f'✅ Pillar 4: Link predictions loaded — {len(df_links)} rows, '
      f'{df_links["anchor_id"].nunique()} anchors')

# ── Pillar 5: Visual similarity scores (placeholder until teammate delivers) ──
P5_PATH = os.path.join(BASE_DIR, 'pillar5_visual_scores.csv')
if os.path.exists(P5_PATH):
    df_visual = pd.read_csv(P5_PATH)
    print(f'✅ Pillar 5: Visual scores loaded — {len(df_visual)} rows')
else:
    df_visual = None
    print('⚠️  Pillar 5: pillar5_visual_scores.csv not found — using default score 0.5')

## 2. Define Constants

In [ ]:
# ── Crime severity weights (aligned with Pillar 2 in CS610_Project.ipynb) ─────
CRIME_SEVERITY = {
    'terrorism':       1.0,
    'homicide':        0.9,
    'weapons':         0.8,
    'narcotics':       0.7,
    'assault':         0.6,
    'financial crime': 0.5,
    'cyber crime':     0.3,
    'Unknown':         0.1,
}

# ── Pillar weights ────────────────────────────────────────────────────────────
W_P1 = 0.40   # Identity Resolution (TF-IDF cosine similarity)
W_P2 = 0.30   # Crime Severity
W_P4 = 0.20   # Hidden Linkage (GraphSAGE)
W_P5 = 0.10   # Visual Similarity (CLIP)

# ── Pillar 4 internal rank weights ────────────────────────────────────────────
LINK_RANK_WEIGHTS = {1: 0.50, 2: 0.30, 3: 0.20}

# ── Final risk thresholds (AML industry norms) ────────────────────────────────
FINAL_THRESHOLDS = {
    'CRITICAL':  0.70,
    'HIGH RISK': 0.50,
    'REVIEW':    0.30,
    'LOW RISK':  0.00,
}

# ── Default visual score when P5 is not available ─────────────────────────────
DEFAULT_VISUAL_SCORE = 0.5

print('Constants loaded.')
print(f'  Pillar weights: P1={W_P1}, P2={W_P2}, P4={W_P4}, P5={W_P5} (sum={W_P1+W_P2+W_P4+W_P5})')
print(f'  P4 rank weights: {LINK_RANK_WEIGHTS} (sum={sum(LINK_RANK_WEIGHTS.values())})')

## 3. Pipeline Functions

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 1 — Identity Resolution
# ═════════════════════════════════════════════════════════════════════════════

def pillar1_identity(client_name: str) -> dict:
    """
    TF-IDF char n-gram cosine similarity against the fugitive database.
    Returns the top match with score and index.
    """
    q = normalize(name_vectorizer.transform([client_name.upper()]))
    sims = cosine_similarity(q, name_embeddings)[0]
    top_idx = int(np.argmax(sims))
    top_score = float(sims[top_idx])
    fugitive = df_unique.iloc[top_idx]
    
    # Top-3 for reporting
    top3_idx = np.argsort(sims)[::-1][:3]
    top3 = [(df_unique.iloc[i]['name'], round(float(sims[i]), 4)) for i in top3_idx]
    
    return {
        'identity_score': top_score,
        'matched_id': fugitive['id'],
        'matched_name': fugitive['name'],
        'matched_crime': fugitive.get('detected_crime_type', 'Unknown'),
        'top3': top3,
        'matched_idx': top_idx,
    }

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 2 — Crime Severity
# ═════════════════════════════════════════════════════════════════════════════

def pillar2_crime_severity(crime_type: str) -> float:
    """
    Look up crime severity score from the CRIME_SEVERITY mapping.
    Returns a float between 0.0 and 1.0.
    """
    return CRIME_SEVERITY.get(crime_type, CRIME_SEVERITY['Unknown'])

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 4 — Hidden Linkage (GraphSAGE)
# ═════════════════════════════════════════════════════════════════════════════

def pillar4_linkage(matched_id: str) -> dict:
    """
    Look up top-3 linked fugitives from pre-computed GraphSAGE predictions.
    
    Formula:
      P4 = sum over rank k in {1,2,3}:
           LINK_RANK_WEIGHTS[k] × link_score_k × crime_severity_k
    
    Returns dict with linkage_score and breakdown details.
    """
    links = df_links[df_links['anchor_id'] == matched_id].sort_values('rank')
    
    if links.empty:
        return {
            'linkage_score': 0.0,
            'linked_fugitives': [],
            'note': 'No pre-computed links found for this fugitive',
        }
    
    breakdown = []
    total = 0.0
    
    for _, row in links.iterrows():
        rank = int(row['rank'])
        link_score = float(row['predicted_link_score'])
        
        # Look up the linked fugitive's crime type
        cand = df_unique[df_unique['id'] == row['candidate_id']]
        if not cand.empty:
            crime_type = cand.iloc[0].get('detected_crime_type', 'Unknown')
        else:
            crime_type = 'Unknown'
        
        severity = pillar2_crime_severity(crime_type)
        rank_weight = LINK_RANK_WEIGHTS.get(rank, 0.0)
        contribution = rank_weight * link_score * severity
        total += contribution
        
        breakdown.append({
            'rank': rank,
            'candidate_name': row['candidate_name'],
            'candidate_id': row['candidate_id'],
            'link_score': round(link_score, 4),
            'crime_type': crime_type,
            'crime_severity': severity,
            'rank_weight': rank_weight,
            'contribution': round(contribution, 4),
        })
    
    return {
        'linkage_score': round(total, 4),
        'linked_fugitives': breakdown,
    }

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PILLAR 5 — Visual Similarity (CLIP)
# ═════════════════════════════════════════════════════════════════════════════

def pillar5_visual(client_name: str, matched_id: str) -> float:
    """
    Look up pre-computed visual similarity score.
    Falls back to DEFAULT_VISUAL_SCORE if not available.
    """
    if df_visual is None:
        return DEFAULT_VISUAL_SCORE
    
    # Try to find a match by client_query and matched_fugitive_id
    match = df_visual[
        (df_visual['client_query'].str.upper() == client_name.upper()) &
        (df_visual['matched_fugitive_id'] == matched_id)
    ]
    
    if not match.empty:
        return float(match.iloc[0]['visual_score'])
    
    # Fallback: try matching by fugitive ID only
    match_by_id = df_visual[df_visual['matched_fugitive_id'] == matched_id]
    if not match_by_id.empty:
        return float(match_by_id.iloc[0]['visual_score'])
    
    return DEFAULT_VISUAL_SCORE

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# FINAL RISK SCORING
# ═════════════════════════════════════════════════════════════════════════════

def classify_risk(score: float) -> tuple:
    """
    Classify final risk score into tiers.
    Returns (tier, action, icon).
    """
    if score >= FINAL_THRESHOLDS['CRITICAL']:
        return ('CRITICAL',
                'Immediate escalation — freeze transaction, notify compliance officer',
                '🔴')
    elif score >= FINAL_THRESHOLDS['HIGH RISK']:
        return ('HIGH RISK',
                'Senior analyst review required within 24 hours',
                '🟠')
    elif score >= FINAL_THRESHOLDS['REVIEW']:
        return ('REVIEW',
                'Junior analyst flag — monitor and log for 30 days',
                '🟡')
    else:
        return ('LOW RISK',
                'Pass — logged to audit trail, no action required',
                '✅')


def screen_client(client_name: str) -> dict:
    """
    Full integrated screening pipeline.
    
    Workflow:
      1. P1: TF-IDF identity resolution → closest fugitive match
      2. P2: Look up matched fugitive's crime severity
      3. P4: Look up top-3 linked fugitives → weighted linkage score
      4. P5: Look up visual similarity score
      5. Final Risk = P1(0.40) + P2(0.30) + P4(0.20) + P5(0.10)
    """
    # ── Pillar 1 ──────────────────────────────────────────────────────────────
    p1 = pillar1_identity(client_name)
    
    # ── Pillar 2 ──────────────────────────────────────────────────────────────
    p2_score = pillar2_crime_severity(p1['matched_crime'])
    
    # ── Pillar 4 ──────────────────────────────────────────────────────────────
    p4 = pillar4_linkage(p1['matched_id'])
    
    # ── Pillar 5 ──────────────────────────────────────────────────────────────
    p5_score = pillar5_visual(client_name, p1['matched_id'])
    
    # ── Final Score ───────────────────────────────────────────────────────────
    final_risk = (
        p1['identity_score'] * W_P1 +
        p2_score             * W_P2 +
        p4['linkage_score']  * W_P4 +
        p5_score             * W_P5
    )
    
    tier, action, icon = classify_risk(final_risk)
    
    return {
        'client_name': client_name,
        'matched_id': p1['matched_id'],
        'matched_name': p1['matched_name'],
        'matched_crime': p1['matched_crime'],
        'top3_matches': p1['top3'],
        'p1_identity_score': round(p1['identity_score'], 4),
        'p2_crime_severity': p2_score,
        'p4_linkage_score': p4['linkage_score'],
        'p4_breakdown': p4['linked_fugitives'],
        'p5_visual_score': round(p5_score, 4),
        'final_risk': round(final_risk, 4),
        'tier': tier,
        'action': action,
        'icon': icon,
    }

## 4. Run Test Cases (from Pillar 1)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# TEST CASES — Aligned with Pillar 1 queries from CS610_Project.ipynb
# ═════════════════════════════════════════════════════════════════════════════

TEST_CASES = [
    # (client_query, description)
    ('S. Nikitenko',              'Initial + surname'),
    ('Hasen Aksema',              'Missing middle name'),
    ('Abdul Sambolotov',          'Single char typo'),
    ('JIAN XIA',                  'Exact match'),
    ('Norbert Bialas',            'Case mismatch'),
    ('Ramazan Chigayev',          'Transliteration variant'),
    ('M. Al-Saeed',               'Abbreviated first name'),
    ('Levitan Elyasov',           'Cross-script: и→i, я→ya'),
    
    # Real INTERPOL fugitive name variants
    ('Jose Solares Gonzalez',     'Missing middle name'),
    ('J. Antonio Solares',        'Initial + dropped surname'),
    ('Gonzalez Jose Antonio',     'Reversed name order'),
    ('Shyamal Rao',               'Missing surname'),
    ('S. Rao Reddy',              'Initial + surname'),
    ('Shyamal Roa Reddy',         'Typo in middle name'),
    ('M. James Winters',          'Initial + surname'),
    ('Marlon Winters',            'Missing middle name'),
    ('Marlon James Wynters',      'Typo in surname'),
]

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# RUN INTEGRATED SCREENING
# ═════════════════════════════════════════════════════════════════════════════

print('=' * 90)
print('  INTEGRATED RISK SCORING PIPELINE')
print('  Formula: P1(0.40) + P2(0.30) + P4(0.20) + P5(0.10)')
print('=' * 90)

all_results = []

for client_name, desc in TEST_CASES:
    result = screen_client(client_name)
    all_results.append(result)
    
    print(f'\n{result["icon"]} {result["tier"]} — Client: "{client_name}" ({desc})')
    print(f'  Matched Fugitive : {result["matched_name"]} [{result["matched_id"]}]')
    print(f'  Matched Crime    : {result["matched_crime"]}')
    print(f'  ┌─────────────────────────────────────────────────────────────')
    print(f'  │ P1 Identity Score  : {result["p1_identity_score"]:.4f}  × {W_P1} = {result["p1_identity_score"]*W_P1:.4f}')
    print(f'  │ P2 Crime Severity  : {result["p2_crime_severity"]:.4f}  × {W_P2} = {result["p2_crime_severity"]*W_P2:.4f}')
    print(f'  │ P4 Linkage Score   : {result["p4_linkage_score"]:.4f}  × {W_P4} = {result["p4_linkage_score"]*W_P4:.4f}')
    print(f'  │ P5 Visual Score    : {result["p5_visual_score"]:.4f}  × {W_P5} = {result["p5_visual_score"]*W_P5:.4f}')
    print(f'  ├─────────────────────────────────────────────────────────────')
    print(f'  │ FINAL RISK SCORE   : {result["final_risk"]:.4f}')
    print(f'  └─────────────────────────────────────────────────────────────')
    print(f'  Action: {result["action"]}')
    
    # P4 Breakdown
    if result['p4_breakdown']:
        print(f'  P4 Linkage Breakdown:')
        for link in result['p4_breakdown']:
            print(f'    Rank {link["rank"]} ({link["rank_weight"]:.0%}): '
                  f'{link["candidate_name"]} — link={link["link_score"]:.4f}, '
                  f'crime={link["crime_type"]} (sev={link["crime_severity"]}), '
                  f'contribution={link["contribution"]:.4f}')
    else:
        print(f'  P4 Linkage: No pre-computed links available')
    
    # Top-3 identity matches
    print(f'  P1 Top-3 Matches:')
    for rank, (name, sc) in enumerate(result['top3_matches'], 1):
        flag = '🔴' if sc >= 0.75 else '🟡' if sc >= 0.50 else '  '
        print(f'    #{rank} {name}: {sc:.4f} {flag}')

## 5. Summary Table

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ═════════════════════════════════════════════════════════════════════════════

summary_data = []
for r in all_results:
    summary_data.append({
        'Client Query': r['client_name'],
        'Matched Fugitive': r['matched_name'],
        'Crime Type': r['matched_crime'],
        'P1 (40%)': r['p1_identity_score'],
        'P2 (30%)': r['p2_crime_severity'],
        'P4 (20%)': r['p4_linkage_score'],
        'P5 (10%)': r['p5_visual_score'],
        'Final Risk': r['final_risk'],
        'Tier': f'{r["icon"]} {r["tier"]}',
    })

df_summary = pd.DataFrame(summary_data)

print('\n' + '=' * 90)
print('  SCREENING RESULTS SUMMARY')
print('=' * 90)
print(df_summary.to_string(index=False))

In [ ]:
# ── Tier Distribution ─────────────────────────────────────────────────────────
from collections import Counter

print('\n' + '─' * 60)
print('  TIER DISTRIBUTION')
print('─' * 60)

tier_counts = Counter(r['tier'] for r in all_results)
for tier_name in ['CRITICAL', 'HIGH RISK', 'REVIEW', 'LOW RISK']:
    count = tier_counts.get(tier_name, 0)
    bar = '█' * count
    print(f'  {tier_name:<12} {bar}  ({count})')

print(f'\n  Total screened: {len(all_results)}')

In [ ]:
# ── Threshold Reference ───────────────────────────────────────────────────────
print('\n' + '─' * 60)
print('  RISK THRESHOLD REFERENCE')
print('─' * 60)
print(f'  {"Tier":<12} {"Threshold":>10}  Action')
print('  ' + '─' * 55)

tier_actions = {
    'CRITICAL':  'Freeze + escalate to compliance officer',
    'HIGH RISK': 'Senior analyst review within 24hr',
    'REVIEW':    'Junior analyst flag, monitor 30 days',
    'LOW RISK':  'Pass, log to audit trail',
}
for tier_name, threshold in FINAL_THRESHOLDS.items():
    print(f'  {tier_name:<12} {threshold:>10.2f}  {tier_actions[tier_name]}')

print(f'\n  Formula: P1×{W_P1} + P2×{W_P2} + P4×{W_P4} + P5×{W_P5}')
print(f'  P4 = (Rank1×0.50 × link_score × severity) + (Rank2×0.30 × ...) + (Rank3×0.20 × ...)')

## 6. Export Results

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# EXPORT
# ═════════════════════════════════════════════════════════════════════════════

export_data = []
for r in all_results:
    row = {
        'client_query': r['client_name'],
        'matched_fugitive_id': r['matched_id'],
        'matched_fugitive_name': r['matched_name'],
        'matched_crime_type': r['matched_crime'],
        'p1_identity_score': r['p1_identity_score'],
        'p2_crime_severity': r['p2_crime_severity'],
        'p4_linkage_score': r['p4_linkage_score'],
        'p5_visual_score': r['p5_visual_score'],
        'final_risk_score': r['final_risk'],
        'risk_tier': r['tier'],
        'action': r['action'],
    }
    
    # Add P4 breakdown columns
    for link in r['p4_breakdown']:
        rank = link['rank']
        row[f'p4_rank{rank}_fugitive'] = link['candidate_name']
        row[f'p4_rank{rank}_link_score'] = link['link_score']
        row[f'p4_rank{rank}_crime'] = link['crime_type']
        row[f'p4_rank{rank}_severity'] = link['crime_severity']
        row[f'p4_rank{rank}_contribution'] = link['contribution']
    
    export_data.append(row)

df_export = pd.DataFrame(export_data)
EXPORT_PATH = os.path.join(OUTPUT_DIR, 'integrated_screening_results.csv')
df_export.to_csv(EXPORT_PATH, index=False)
print(f'✅ Results exported to: {EXPORT_PATH}')
print(f'   Rows: {len(df_export)}, Columns: {len(df_export.columns)}')